# Human vs AI Trivia Challenge - WITH FINE-TUNING

**dc_dalin - Week 6 Contribution**

Compare human vs AI performance on trivia questions, but now with **Base GPT-4o-mini vs Fine-Tuned GPT-4o-mini**!

This demonstrates the value of fine-tuning a closed-source model.

In [ ]:
import os
import json
import time
import random
from datetime import datetime
from typing import Dict, List, Tuple

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

load_dotenv()
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

## Load Fine-Tuned Model ID

Make sure you've run `fine_tune_setup.ipynb` first to create the fine-tuned model.

In [ ]:
# Load the fine-tuned model ID
try:
    with open('fine_tuned_model_id.txt', 'r') as f:
        FINE_TUNED_MODEL = f.read().strip()
    print(f"✅ Loaded fine-tuned model: {FINE_TUNED_MODEL}")
except FileNotFoundError:
    print("❌ Fine-tuned model ID not found!")
    print("Please run 'fine_tune_setup.ipynb' first to create the fine-tuned model.")
    print("\nOr manually set the model ID below:")
    FINE_TUNED_MODEL = "ft:gpt-4o-mini-2024-07-18:your-org:trivia-expert:xxxxx"  # Replace with your model ID

BASE_MODEL = "gpt-4o-mini"

In [ ]:
# Load questions
with open('questions_dataset.json', 'r') as f:
    data = json.load(f)
    questions = data['questions']
    metadata = data['metadata']

print(f"Loaded {len(questions)} questions | Author: {metadata['author']}")

## Updated AI Answer Function

Now supports both base and fine-tuned models for comparison.

In [ ]:
def get_ai_answer(question: str, choices: List[str], model: str, timeout: int = 5) -> Tuple[str, float, bool]:
    """Get AI answer using specified model."""
    start_time = time.time()
    prompt = f"""Answer this trivia question with just the answer:

Question: {question}

Choices:
{chr(10).join([f'{chr(65+i)}. {choice}' for i, choice in enumerate(choices)])}

Provide only the answer."""
    
    try:
        response = client.chat.completions.create(
            model=model,
            max_tokens=50,
            timeout=timeout,
            temperature=0,
            messages=[{"role": "user", "content": prompt}]
        )
        elapsed = (time.time() - start_time) * 1000
        return response.choices[0].message.content.strip(), elapsed, True
    except Exception as e:
        return f"Error: {str(e)}", (time.time() - start_time) * 1000, False

def check_answer(given: str, correct: str) -> bool:
    """Check if answer is correct with fuzzy matching."""
    given, correct = given.lower().strip(), correct.lower().strip()
    if given == correct or correct in given or given in correct:
        return True
    
    # Fuzzy matching
    given_words = set(w for w in given.split() if w not in ['the', 'a', 'an', 'of', 'and'])
    correct_words = set(w for w in correct.split() if w not in ['the', 'a', 'an', 'of', 'and'])
    
    if correct_words and len(given_words.intersection(correct_words)) / len(correct_words) >= 0.7:
        return True
    return False

## Game Logic with Model Comparison

In [ ]:
# Game state
game_state = {
    'current_idx': 0,
    'questions': [],
    'results': [],
    'human_score': 0,
    'base_ai_score': 0,
    'finetuned_ai_score': 0,
    'num_questions': 10
}

def start_game(num: int, difficulty: str, category: str) -> Tuple:
    filtered = questions.copy()
    if difficulty != "All":
        filtered = [q for q in filtered if q['difficulty'] == difficulty]
    if category != "All":
        filtered = [q for q in filtered if q['category'] == category]
    
    if len(filtered) < num:
        return (
            f"Only {len(filtered)} questions available. Adjust filters.",
            gr.update(visible=False), gr.update(visible=True),
            "", "", "", "", gr.update(visible=False)
        )
    
    game_state.update({
        'current_idx': 0,
        'questions': random.sample(filtered, num),
        'results': [],
        'human_score': 0,
        'base_ai_score': 0,
        'finetuned_ai_score': 0,
        'num_questions': num
    })
    
    q = game_state['questions'][0]
    return (
        f"Question 1/{num}",
        gr.update(visible=True),
        gr.update(visible=False),
        q['question'],
        gr.Radio(choices=q['choices'], label="Your answer:"),
        f"**{q['category']}** | {q['difficulty']}",
        "",
        gr.update(visible=True)
    )

def submit_answer(human_answer: str) -> Tuple:
    if not human_answer:
        return "Select an answer!", "", gr.update(), gr.update()
    
    idx = game_state['current_idx']
    q = game_state['questions'][idx]
    
    # Get human result
    human_correct = check_answer(human_answer, q['answer'])
    
    # Get base model result
    base_answer, base_time, base_success = get_ai_answer(q['question'], q['choices'], BASE_MODEL)
    base_correct = check_answer(base_answer, q['answer']) if base_success else False
    
    # Get fine-tuned model result
    ft_answer, ft_time, ft_success = get_ai_answer(q['question'], q['choices'], FINE_TUNED_MODEL)
    ft_correct = check_answer(ft_answer, q['answer']) if ft_success else False
    
    # Update scores
    if human_correct:
        game_state['human_score'] += 1
    if base_correct:
        game_state['base_ai_score'] += 1
    if ft_correct:
        game_state['finetuned_ai_score'] += 1
    
    # Store results
    game_state['results'].append({
        'question': q['question'],
        'category': q['category'],
        'difficulty': q['difficulty'],
        'correct_answer': q['answer'],
        'human_answer': human_answer,
        'human_correct': human_correct,
        'base_ai_answer': base_answer,
        'base_ai_correct': base_correct,
        'base_ai_time_ms': base_time,
        'finetuned_ai_answer': ft_answer,
        'finetuned_ai_correct': ft_correct,
        'finetuned_ai_time_ms': ft_time
    })
    
    feedback = f"""
**Correct Answer**: {q['answer']}

**You**: {human_answer} {'✅' if human_correct else '❌'}  
**Base AI**: {base_answer} {'✅' if base_correct else '❌'} ({base_time:.0f}ms)  
**Fine-Tuned AI**: {ft_answer} {'✅' if ft_correct else '❌'} ({ft_time:.0f}ms)

**Score**: Human {game_state['human_score']} | Base AI {game_state['base_ai_score']} | Fine-Tuned AI {game_state['finetuned_ai_score']}

*{q['explanation']}*
"""
    
    is_last = idx + 1 >= game_state['num_questions']
    return (
        feedback,
        "",
        gr.update(visible=False),
        gr.update(visible=True, value="View Results" if is_last else "Next ➡️")
    )

def next_question() -> Tuple:
    game_state['current_idx'] += 1
    idx = game_state['current_idx']
    
    # Check if this was the last question
    if idx >= game_state['num_questions']:
        # Show results
        h_score = game_state['human_score']
        base_score = game_state['base_ai_score']
        ft_score = game_state['finetuned_ai_score']
        total = game_state['num_questions']
        
        # Determine winner
        scores = [(h_score, "🏆 Human"), (base_score, "🤖 Base AI"), (ft_score, "⚡ Fine-Tuned AI")]
        max_score = max(s[0] for s in scores)
        winners = [name for score, name in scores if score == max_score]
        
        if len(winners) > 1:
            winner = "🤝 Tie: " + " & ".join(winners)
        else:
            winner = winners[0] + " Wins!"
        
        df = pd.DataFrame(game_state['results'])
        
        # Calculate average response times
        base_avg_time = df['base_ai_time_ms'].mean()
        ft_avg_time = df['finetuned_ai_time_ms'].mean()
        
        # Build category table
        cat_stats = df.groupby('category')[['human_correct', 'base_ai_correct', 'finetuned_ai_correct']].sum()
        cat_table = "\n".join([
            f"- **{cat}**: Human {int(row['human_correct'])} | Base AI {int(row['base_ai_correct'])} | Fine-Tuned {int(row['finetuned_ai_correct'])}" 
            for cat, row in cat_stats.iterrows()
        ])
        
        # Calculate improvement
        improvement = ft_score - base_score
        improvement_pct = ((ft_score / total) - (base_score / total)) * 100
        
        summary = f"""
# {winner}

## Final Scores
**Human**: {h_score}/{total} ({h_score/total*100:.1f}%)  
**Base AI**: {base_score}/{total} ({base_score/total*100:.1f}%)  
**Fine-Tuned AI**: {ft_score}/{total} ({ft_score/total*100:.1f}%)  

## Fine-Tuning Impact
**Improvement**: {improvement:+d} questions ({improvement_pct:+.1f}%)  
**Base Avg Response**: {base_avg_time:.0f}ms  
**Fine-Tuned Avg Response**: {ft_avg_time:.0f}ms  

### Performance by Category
{cat_table}

---
*dc_dalin - Week 6 - Demonstrating Closed-Source Model Fine-Tuning*
"""
        
        fig = create_charts(df, h_score, base_score, ft_score, total)
        
        # Save results
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        with open(f"results_finetuned_{timestamp}.json", 'w') as f:
            json.dump({
                'scores': {
                    'human': h_score, 
                    'base_ai': base_score, 
                    'finetuned_ai': ft_score
                },
                'models': {
                    'base': BASE_MODEL,
                    'finetuned': FINE_TUNED_MODEL
                },
                'results': game_state['results']
            }, f, indent=2)
        
        return (
            "",
            "",
            gr.update(choices=[], value=None),
            "",
            "",
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=True),
            summary,
            fig
        )
    
    # Load next question
    q = game_state['questions'][idx]
    return (
        f"Question {idx + 1}/{game_state['num_questions']}",
        q['question'],
        gr.Radio(choices=q['choices'], label="Your answer:", value=None),
        f"**{q['category']}** | {q['difficulty']}",
        "",
        gr.update(visible=True),
        gr.update(visible=False),
        gr.update(visible=True),
        gr.update(visible=False),
        "",
        None
    )

## Enhanced Visualization

In [ ]:
def create_charts(df: pd.DataFrame, h: int, base: int, ft: int, total: int):
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "Overall Accuracy Comparison", 
            "Performance by Category", 
            "Correct Answer Distribution", 
            "Response Time Comparison"
        ),
        specs=[[{"type": "bar"}, {"type": "bar"}], [{"type": "pie"}, {"type": "bar"}]]
    )
    
    # Overall accuracy
    fig.add_trace(go.Bar(
        x=["Human", "Base AI", "Fine-Tuned AI"], 
        y=[h/total*100, base/total*100, ft/total*100],
        marker_color=["#3498db", "#e74c3c", "#2ecc71"],
        text=[f"{h}/{total}", f"{base}/{total}", f"{ft}/{total}"],
        textposition='auto'
    ), row=1, col=1)
    
    # By category
    cat_stats = df.groupby('category')[['human_correct', 'base_ai_correct', 'finetuned_ai_correct']].sum()
    fig.add_trace(go.Bar(
        name="Human", 
        x=cat_stats.index, 
        y=cat_stats['human_correct'],
        marker_color="#3498db"
    ), row=1, col=2)
    fig.add_trace(go.Bar(
        name="Base AI", 
        x=cat_stats.index, 
        y=cat_stats['base_ai_correct'],
        marker_color="#e74c3c"
    ), row=1, col=2)
    fig.add_trace(go.Bar(
        name="Fine-Tuned AI", 
        x=cat_stats.index, 
        y=cat_stats['finetuned_ai_correct'],
        marker_color="#2ecc71"
    ), row=1, col=2)
    
    # Distribution pie chart
    all_correct = sum(df['human_correct'] & df['base_ai_correct'] & df['finetuned_ai_correct'])
    ft_only = sum(df['finetuned_ai_correct'] & ~df['base_ai_correct'])
    base_only = sum(df['base_ai_correct'] & ~df['finetuned_ai_correct'])
    none_correct = total - df[['human_correct', 'base_ai_correct', 'finetuned_ai_correct']].any(axis=1).sum()
    
    fig.add_trace(go.Pie(
        labels=["All Correct", "Fine-Tuned Only", "Base Only", "None"],
        values=[all_correct, ft_only, base_only, none_correct],
        marker_colors=["#2ecc71", "#f39c12", "#e74c3c", "#95a5a6"]
    ), row=2, col=1)
    
    # Response times
    avg_base_time = df['base_ai_time_ms'].mean()
    avg_ft_time = df['finetuned_ai_time_ms'].mean()
    
    fig.add_trace(go.Bar(
        x=["Base AI", "Fine-Tuned AI"],
        y=[avg_base_time, avg_ft_time],
        marker_color=["#e74c3c", "#2ecc71"],
        text=[f"{avg_base_time:.0f}ms", f"{avg_ft_time:.0f}ms"],
        textposition='auto'
    ), row=2, col=2)
    
    fig.update_layout(
        height=900, 
        showlegend=True,
        title_text="Fine-Tuning Performance Analysis (dc_dalin)",
        title_x=0.5
    )
    fig.update_yaxes(title_text="Accuracy (%)", row=1, col=1)
    fig.update_yaxes(title_text="Correct Answers", row=1, col=2)
    fig.update_yaxes(title_text="Avg Time (ms)", row=2, col=2)
    
    return fig

## Gradio Interface

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title="Human vs AI Trivia - With Fine-Tuning") as demo:
    gr.Markdown("""
    # 🎯 Human vs Base AI vs Fine-Tuned AI Trivia
    
    **Demonstrating the value of fine-tuning a closed-source model (OpenAI GPT-4o-mini)**
    
    *dc_dalin - Week 6*
    """)
    
    with gr.Row(visible=True) as setup:
        with gr.Column():
            num = gr.Slider(5, 25, value=10, step=5, label="Questions")
            diff = gr.Dropdown(["All", "Easy", "Medium", "Hard"], value="All", label="Difficulty")
            cat = gr.Dropdown(["All"] + metadata['categories'], value="All", label="Category")
            
            gr.Markdown(f"""
            ### Models Being Compared:
            - **Base Model**: `{BASE_MODEL}`
            - **Fine-Tuned Model**: `{FINE_TUNED_MODEL[:50]}...`
            """)
            
            start_btn = gr.Button("Start Challenge", variant="primary")
    
    status = gr.Markdown("")
    
    with gr.Row(visible=False) as game:
        with gr.Column():
            q_text = gr.Markdown("")
            q_meta = gr.Markdown("")
            choices = gr.Radio(label="Your answer:", choices=[])
            submit_btn = gr.Button("Submit", variant="primary", visible=True)
            next_btn = gr.Button("Next ➡️", variant="secondary", visible=False)
            feedback = gr.Markdown("")
    
    with gr.Row(visible=False) as results:
        with gr.Column():
            results_text = gr.Markdown("")
            chart = gr.Plot()
            restart = gr.Button("Play Again", variant="primary")
    
    # Event handlers
    start_btn.click(
        start_game, 
        [num, diff, cat], 
        [status, game, setup, q_text, choices, q_meta, feedback, submit_btn]
    )
    
    submit_btn.click(
        submit_answer, 
        [choices], 
        [feedback, status, submit_btn, next_btn]
    )
    
    next_btn.click(
        next_question, 
        outputs=[status, q_text, choices, q_meta, feedback, submit_btn, next_btn, game, results, results_text, chart]
    )
    
    restart.click(
        lambda: (gr.update(visible=True), gr.update(visible=False)), 
        outputs=[setup, results]
    )

demo.launch(share=False)

## Quick Batch Evaluation (Optional)

Run all questions through both models for a comprehensive comparison.

In [ ]:
def batch_evaluate(num_questions=50):
    """Evaluate both models on a batch of questions."""
    print(f"Evaluating {num_questions} questions...\n")
    
    test_questions = random.sample(questions, min(num_questions, len(questions)))
    
    base_correct = 0
    ft_correct = 0
    base_times = []
    ft_times = []
    
    for i, q in enumerate(test_questions, 1):
        print(f"\rProcessing {i}/{num_questions}...", end="")
        
        # Base model
        base_answer, base_time, base_success = get_ai_answer(
            q['question'], q['choices'], BASE_MODEL
        )
        if base_success and check_answer(base_answer, q['answer']):
            base_correct += 1
        base_times.append(base_time)
        
        # Fine-tuned model
        ft_answer, ft_time, ft_success = get_ai_answer(
            q['question'], q['choices'], FINE_TUNED_MODEL
        )
        if ft_success and check_answer(ft_answer, q['answer']):
            ft_correct += 1
        ft_times.append(ft_time)
    
    print("\n\n" + "=" * 60)
    print("📊 BATCH EVALUATION RESULTS")
    print("=" * 60)
    print(f"\nBase Model ({BASE_MODEL}):")
    print(f"  Accuracy: {base_correct}/{num_questions} ({base_correct/num_questions*100:.1f}%)")
    print(f"  Avg Response Time: {sum(base_times)/len(base_times):.0f}ms")
    
    print(f"\nFine-Tuned Model:")
    print(f"  Accuracy: {ft_correct}/{num_questions} ({ft_correct/num_questions*100:.1f}%)")
    print(f"  Avg Response Time: {sum(ft_times)/len(ft_times):.0f}ms")
    
    improvement = ft_correct - base_correct
    improvement_pct = (ft_correct - base_correct) / num_questions * 100
    
    print(f"\n🎯 Fine-Tuning Impact:")
    print(f"  Improvement: {improvement:+d} questions ({improvement_pct:+.1f}%)")
    print("=" * 60)

# Uncomment to run batch evaluation
# batch_evaluate(num_questions=50)

## Summary

This notebook demonstrates:

1. ✅ **Using a fine-tuned closed-source model** (OpenAI GPT-4o-mini)
2. ✅ **Direct comparison** between base and fine-tuned models
3. ✅ **Performance metrics** showing the impact of fine-tuning
4. ✅ **Interactive evaluation** through the Gradio interface

### Key Takeaways:

- Fine-tuning improves model performance on domain-specific tasks
- OpenAI's fine-tuning API makes it easy to customize closed-source models
- The comparison shows measurable improvements in accuracy
- Response times remain comparable between base and fine-tuned models

---

**dc_dalin - Week 6 Contribution**
